# Diffusion Batch Inference on Downloads Clips

This notebook resolves the most recent diffusion run automatically, snapshots its `latest.pt` checkpoint if needed, then generates a large batch of random short-form transfers from songs in `Downloads/`.

Outputs:
- source excerpts
- generated diffusion outputs
- `manifest.csv` with source, target, offset, run, and checkpoint metadata
- `summary.json` with the resolved run and output folder


In [ ]:
from pathlib import Path
import importlib
import json
import sys

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'dggr').exists() and (path / 'lab 3').exists() and (path / 'lab 3.1').exists():
            return path
    raise RuntimeError('Could not resolve repo root from current working directory.')

REPO = find_repo_root(Path.cwd().resolve())
SCRIPTS = REPO / 'lab 3.1' / 'scripts'
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

import diffusion_downloads_batch as ddb
importlib.reload(ddb)
REPO

In [ ]:
cfg = ddb.DiffusionDownloadsBatchConfig(
    downloads_dir=Path.home() / 'Downloads',
    run_dir=None,               # None = auto-pick most recent diffusion run
    checkpoint_path=None,      # single-checkpoint mode only
    cache_dir=None,            # None = infer from run config or fallback cache
    n_clips=30,
    clip_seconds=3.0,
    n_frames=256,
    ddim_steps=50,
    guidance_scale=2.0,
    t_start=320,
    style_strength=0.90,
    device='auto',
    seed=328,
)
RUN_ALL = True

ctx = ddb.resolve_inference_context(cfg)
print('Resolved run dir:      ', ctx['run_dir'])
print('Resolved checkpoint:   ', ctx['checkpoint_path'])
print('Resolved cache dir:    ', ctx['cache_dir'])
print('Planned output root:   ', cfg.output_root / cfg.tag)

In [ ]:
checkpoint_panel = ddb.choose_checkpoint_panel(
    ctx['run_dir'],
    include_latest=True,
    include_best=True,
    include_epoch6=True,
    n_random_epochs=3,
    seed=cfg.seed,
)
pd.DataFrame([
    {'label': row['label'], 'path': str(row['path'])}
    for row in checkpoint_panel
])

In [ ]:
jobs = ddb.plan_jobs(cfg)
preview = pd.DataFrame(jobs)
display(preview.head(12))
print('Total planned clips:', len(preview))
print('Target counts:')
display(preview['target_genre'].value_counts().sort_index())

In [ ]:
summary = None
if RUN_ALL:
    summary = ddb.run_multi_checkpoint_inference(cfg, checkpoint_panel)
    print(json.dumps(summary, indent=2, default=str))
else:
    print('Set RUN_ALL = True to generate the batch inference clips across all checkpoints in the panel.')

In [ ]:
summary_path = cfg.output_root / cfg.tag / 'summary.json'
manifest_path = cfg.output_root / cfg.tag / 'manifest.csv'
if summary_path.exists():
    print(summary_path)
    print(manifest_path)
    display(pd.read_csv(manifest_path).head(10))
else:
    print('No outputs yet for this tag.')